In [1]:
import whisperx
from os import environ
from whisperx.diarize import DiarizationPipeline

hf_token = environ.get('hugging_token')
model_save_path=environ.get('model_save_path')

def transcribe(
    audio_file_path: str,
    compute: str = 'fast',
    language='ru',
    model: str = 'medium',
    diarization: bool = False,
    min_speakers: int = None,
    max_speakers: int = None,
    token: str = None,
):
    """
    compute_type: fast|accurate
    Recommended Settings for CPU 1 or 8

    model:small|medium|large
    """
    if False:
        pass
    else:
        print('🐌 Running on CPU...')
        device = 'cpu'
        batch_size = 1
        if compute == 'fast':
            compute_type = 'int8'
        else:
            compute_type = 'float32'
    model = whisperx.load_model(model, compute_type=compute_type, language=language, device=device,download_root=model_save_path)
    audio = whisperx.load_audio(audio_file_path)
    result = model.transcribe(audio, batch_size=batch_size)

    if diarization:
        # 2. Выравнивание таймингов слов (важно для точности склейки со спикером)
        print('⏳ Этап 2: Точное выравнивание временных меток...')
        model_a, metadata = whisperx.load_align_model(language_code=result['language'], device=device)
        result = whisperx.align(
            result['segments'],
            model_a,
            metadata,
            audio,
            device,
            return_char_alignments=False,
        )

        # 3. Диаризация (Pyannote)
        print('⏳ Этап 3: Разделение спикеров (Pyannote)...')
        diarize_model = DiarizationPipeline(token=token, device=device)

        # Опционально: если вы точно знаете число спикеров, добавьте: min_speakers=2, max_speakers=2
        if min_speakers is not None and max_speakers is not None:
            diarize_segments = diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)
        else:
            diarize_segments = diarize_model(audio)

        # 4. Сопоставление текста и спикеров
        print('⏳ Этап 4: Объединение текста и голосов...')
        result = whisperx.assign_word_speakers(diarize_segments, result)

    return result["segments"]


#r = transcribe(file_name, model='small', token=hf_token)

In [2]:
%%time
file_name = 'Recording 20260513134530.m4a'

r=transcribe(file_name,model='small',diarization=False)

🐌 Running on CPU...
2026-05-24 03:01:18 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.4. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../opt/conda/lib/python3.11/site-packages/whisperx/assets/pytorch_model.bin`


CPU times: user 2min 38s, sys: 21 s, total: 2min 59s
Wall time: 48 s


In [ ]:
    for segment in r["segments"]:
        speaker = segment.get("speaker", "UNKNOWN_SPEAKER")
        print(f"[{segment['start']:.2f}s -> {segment['end']:.2f}s] {speaker}: {segment['text']}")